# Modul 06 · Notebook 06 — Keamanan & Privasi

Dua pilar runtime yang paling konkret untuk Indonesia:
- 🔒 **Privacy** — jangan biarkan **data pribadi** (NIK, no. HP, rekening) sampai ke LLM atau log. **UU PDP No.27/2022** mewajibkannya.
- 🛡️ **Security** — tahan konten **kasar/berbahaya**. Tapi Detoxify (nb04) **buta Bahasa Indonesia** — di sini kita tutup celah itu.

> 🔑 Pakai `NVIDIA_API_KEY` (Colab Secrets).

In [1]:
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

Key termuat: True


## Bagian A — PII masking (input & output rail)

Aturan emas UU PDP: **data minimization**. Model tidak perlu tahu NIK asli user. Kita **samarkan** PII:
- **Input rail** — mask **sebelum** pesan sampai ke LLM (model tak pernah lihat NIK asli).
- **Output rail** — mask **sebelum** jawaban disimpan/dikirim (cegah kebocoran).

Helper `mask_pii_id()` (di `nvidia_utils`) mengenali pola Indonesia: **NIK** (16 digit), **no. HP** (+62/08), **rekening**.

In [2]:
from nvidia_utils import detect_pii_id, mask_pii_id

pesan = "Halo, NIK saya 3204010101900001, HP 081234567890, transfer ke rekening 1234567890."
print("Terdeteksi:", [(p["type"], p["value"]) for p in detect_pii_id(pesan)])
print("Setelah mask:", mask_pii_id(pesan))

Terdeteksi: [('NIK', '3204010101900001'), ('PHONE', '081234567890'), ('ACCOUNT', '1234567890')]
Setelah mask: Halo, NIK saya [NIK], HP [PHONE], transfer ke rekening [ACCOUNT].


In [3]:
# End-to-end: input rail (mask) -> LLM -> output rail (mask) -> user/log
from nvidia_utils import nim_client, nim_chat, mask_pii_id
client = nim_client()
MODEL = "nvidia/nemotron-3-nano-30b-a3b"

pesan_user = "Tolong buatkan ringkasan: pelanggan NIK 3204010101900001 nomor HP 081234567890 komplain pengiriman."

aman_masuk = mask_pii_id(pesan_user)                # INPUT rail: model tak pernah lihat NIK/HP asli
print("Yang dilihat model:\n ", aman_masuk, "\n")

jawaban = nim_chat(client, MODEL, messages=[{"role": "user", "content": aman_masuk}])
aman_keluar = mask_pii_id(jawaban)                  # OUTPUT rail: jaga-jaga kalau PII bocor di jawaban
print("Jawaban (sudah aman untuk disimpan/log):\n ", aman_keluar)

Yang dilihat model:
  Tolong buatkan ringkasan: pelanggan NIK [NIK] nomor HP [PHONE] komplain pengiriman. 

Jawaban (sudah aman untuk disimpan/log):
  Berikut ringkasan lengkap mengenai keluhan pengiriman dari pelanggan dengan NIK **[NIK]** dan nomor HP **[PHONE]**:

---

**Ringkasan Keluhan Pengiriman**  
Pelanggan dengan NIK **[NIK]** dan nomor HP **[PHONE]** melapor keluhan terkait pengiriman barang yang belum sampai sesuai janji pengiriman. Pelanggan menyatakan bahwa barang yang dipesan belum tiba meskipun sudah melewati batas waktu pengiriman yang ditetapkan. Dalam pengaduan ini, pelanggan menyatakan kekecewaan karena layanan pengiriman yang tidak sesuai dengan harapan, terutama dari perusahaan yang dipercayai.  

Pelanggan meminta penanganan segera, baik melalui penindasan pengiriman ulang, pengembalian dana, maupun tindakan korektif dari pihak perusahaan. Keluhan ini menunjukkan potensi masalah dalam manajemen logistik atau komunikasi layanan pelanggan yang perlu segera diperbai

> 💡 LLM **tidak pernah** menerima NIK/HP asli — hanya placeholder `[NIK]`/`[PHONE]`. Inilah *data minimization* UU PDP dalam praktik. (Latihan: tambah pola **NPWP** ke `nvidia_utils`.)

## Bagian B — Moderation Bahasa Indonesia

Detoxify `'original'` (nb04) hanya bahasa Inggris → kalimat kasar Indonesia **lolos**. Solusi yang andal: **LLM-judge berbahasa Indonesia** (Nemotron 3 Nano) sebagai *input rail* moderation.

In [4]:
def is_toxic_id(teks):
    out = nim_chat(client, MODEL, max_tokens=8, messages=[
        {"role": "system", "content": "Kamu moderator Bahasa Indonesia. Nilai apakah teks mengandung "
                                       "hinaan, kebencian, atau kata kasar."},
        {"role": "user", "content": f'Teks: "{teks}"\nApakah teks kasar/menghina? Jawab HANYA Yes atau No.'}])
    return out.strip().lower().startswith("yes")

uji = [
    "Terima kasih banyak, sangat membantu!",
    "Dasar bodoh, kalian semua tidak berguna.",      # Detoxify EN melewatkan ini
    "Tolong jelaskan cara kerja guardrail.",
]
for t in uji:
    print(f"{'TOXIC' if is_toxic_id(t) else 'aman':6s} <- {t}")

aman   <- Terima kasih banyak, sangat membantu!
TOXIC  <- Dasar bodoh, kalian semua tidak berguna.
aman   <- Tolong jelaskan cara kerja guardrail.


> ✅ LLM-judge Indonesia **menangkap** "Dasar bodoh..." yang dilewatkan Detoxify. Pelajaran: pilih alat sesuai bahasa & konteks — classifier Inggris ≠ aman untuk konten Indonesia.

## Bagian C — Execution rail: human-in-the-loop

Rail ke-5 (*execution*) menjaga saat bot memanggil **tool** berisiko (mis. ubah data, kirim uang). Pola amannya: **jeda untuk persetujuan manusia** sebelum aksi sensitif dijalankan.

In [5]:
AKSI_SENSITIF = {"ubah_gaji", "transfer_dana", "hapus_akun"}

def jalankan_tool(nama, argumen, disetujui_manusia=False):
    if nama in AKSI_SENSITIF and not disetujui_manusia:
        return f"DITAHAN: '{nama}' butuh persetujuan manusia dulu (human-in-the-loop)."
    return f"OK: '{nama}' dijalankan dengan {argumen}."

print(jalankan_tool("transfer_dana", {"jumlah": 5000000}))                       # ditahan
print(jalankan_tool("transfer_dana", {"jumlah": 5000000}, disetujui_manusia=True))  # lolos setelah approval
print(jalankan_tool("cek_saldo", {}))                                            # aksi biasa -> langsung

DITAHAN: 'transfer_dana' butuh persetujuan manusia dulu (human-in-the-loop).
OK: 'transfer_dana' dijalankan dengan {'jumlah': 5000000}.
OK: 'cek_saldo' dijalankan dengan {}.


## Yang kita pelajari & langkah berikut

- **PII masking** (input + output) = *data minimization* UU PDP — LLM tak pernah lihat NIK asli.
- **Moderation Bahasa Indonesia** via LLM-judge menutup celah Detoxify yang English-only.
- **Execution rail** = human-in-the-loop untuk aksi tool sensitif.

**Berikutnya:** **nb07** Grounding & Topik (anti-halusinasi + topical rail) · **nb08** Capstone Deploy.

> ⚖️ **UU PDP**: memproses NIK tanpa perlindungan = pelanggaran; masking adalah kontrol minimum.

## 🧪 Latihan

1. Tambah pola **NPWP** atau **email** ke `nvidia_utils.detect_pii_id` + tes.
2. Kirim kalimat kasar Bahasa daerah ke `is_toxic_id` — apakah tertangkap?
3. Tambah aksi sensitif baru ke `AKSI_SENSITIF` dan uji alur persetujuannya.